In [3]:


# ---------------------------------------------------------------
# Thresholds: (warning_start, critical_start)
#   value <  warning_start            -> NORMAL
#   warning_start <= value <= critical_start -> WARNING
#   value >  critical_start           -> CRITICAL
# ---------------------------------------------------------------
THRESHOLDS = {
    "Temperature": {"warning": 80, "critical": 100, "unit": "°C"},
    "Vibration": {"warning": 5, "critical": 8, "unit": "mm/s"},
}

NORMAL, WARNING, CRITICAL = "NORMAL", "WARNING", "CRITICAL"
SEVERITY_ORDER = {NORMAL: 0, WARNING: 1, CRITICAL: 2}


def validate_reading(value):
    """Return a float if the reading is a valid non-negative number, else raise ValueError."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        raise ValueError(f"Invalid reading: {value!r} is not a number.")
    if number != number:  # NaN check
        raise ValueError("Invalid reading: value is missing (NaN).")
    if number < 0:
        raise ValueError(f"Invalid reading: {number} cannot be negative.")
    return number


def classify_parameter(parameter, value):
    """Classify one reading as NORMAL, WARNING or CRITICAL."""
    if parameter not in THRESHOLDS:
        raise KeyError(f"Unknown parameter: {parameter}")
    value = validate_reading(value)
    limits = THRESHOLDS[parameter]
    if value > limits["critical"]:
        return CRITICAL
    if value >= limits["warning"]:
        return WARNING
    return NORMAL


def condition_message(parameter, value):
    """Return the status and a human-readable message for one reading."""
    status = classify_parameter(parameter, value)
    unit = THRESHOLDS[parameter]["unit"]
    if status == NORMAL:
        msg = "NORMAL: Equipment operating within defined monitoring limits."
    elif status == WARNING:
        msg = f"WARNING: Equipment {parameter.lower()} approaching predefined threshold."
    else:
        msg = (f"CRITICAL: Equipment {parameter.lower()} exceeds predefined threshold. "
               "Inspection required")
    return status, msg


def evaluate_equipment(equipment_id, temperature, vibration):
    """
    Check temperature and vibration of one piece of equipment.
    Returns a dict with the overall status and a result per parameter.
    """
    readings = {"Temperature": temperature, "Vibration": vibration}
    results = []
    overall = NORMAL
    for parameter, value in readings.items():
        status, msg = condition_message(parameter, value)
        results.append({
            "equipment_id": equipment_id,
            "parameter": parameter,
            "value": float(value),
            "unit": THRESHOLDS[parameter]["unit"],
            "status": status,
            "message": msg,
        })
        if SEVERITY_ORDER[status] > SEVERITY_ORDER[overall]:
            overall = status
    return {"equipment_id": equipment_id, "overall_status": overall, "results": results}


def format_alert(result):
    """Format one NON-NORMAL result like Figure 1 in the project brief."""
    bar = "=" * 53
    if result["status"] == CRITICAL:
        threshold_text = "Above Critical Threshold"
        action = "Inspect equipment and initiate appropriate\n maintenance/safety procedures."
    else:
        threshold_text = "Approaching Critical Threshold"
        action = "Monitor closely and schedule an inspection."
    return (
        f"{bar}\n"
        f"MINING MONITORING ALERT\n"
        f"{bar}\n"
        f"[{result['status']}]\n"
        f"Equipment ID: {result['equipment_id']}\n"
        f"Parameter: {result['parameter']}\n"
        f"Value: {result['value']} {result['unit']}\n"
        f"Status: {threshold_text}\n"
        f"Recommended Action:\n {action}\n"
        f"{bar}"
    )


def generate_alerts(evaluation):
    """Return a list of formatted alert strings for every non-normal parameter."""
    return [format_alert(r) for r in evaluation["results"] if r["status"] != NORMAL]


def monitor_dataframe(df):
    """
    Evaluate a whole pandas DataFrame.
    Needs columns: 'Equipment ID', 'Temperature', 'Vibration'.
    Returns a copy with extra columns: Temp Status, Vib Status, Condition.
    Rows with bad readings get 'INVALID' instead of crashing the program.
    """
    out = df.copy()
    temp_status, vib_status, overall = [], [], []
    for _, row in out.iterrows():
        try:
            ev = evaluate_equipment(row["Equipment ID"], row["Temperature"], row["Vibration"])
            temp_status.append(ev["results"][0]["status"])
            vib_status.append(ev["results"][1]["status"])
            overall.append(ev["overall_status"])
        except ValueError:
            temp_status.append("INVALID")
            vib_status.append("INVALID")
            overall.append("INVALID")
    out["Temp Status"] = temp_status
    out["Vib Status"] = vib_status
    out["Condition"] = overall
    return out


def condition_summary(monitored_df):
    """Count of equipment per condition + % operating normally (for the dashboard)."""
    counts = monitored_df["Condition"].value_counts().to_dict()
    total = len(monitored_df)
    normal = counts.get(NORMAL, 0)
    return {
        "total": total,
        "normal": normal,
        "warning": counts.get(WARNING, 0),
        "critical": counts.get(CRITICAL, 0),
        "invalid": counts.get("INVALID", 0),
        "percent_normal": round(normal / total * 100, 1) if total else 0.0,
    }


# ---------------------------------------------------------------
# Quick self-test: run `python equipment_condition.py`
# ---------------------------------------------------------------
if __name__ == "__main__":
    import pandas as pd

    sample = pd.DataFrame({
        "Equipment ID": ["TRK-001", "TRK-012", "LDR-004", "CNV-002"],
        "Temperature": [72, 91, 105, 60],
        "Vibration": [3.1, 9.2, 4.0, 6.5],
    })
    result = monitor_dataframe(sample)
    print(result.to_string(index=False))
    print()
    print(condition_summary(result))
    print()
    for _, row in sample.iterrows():
        for alert in generate_alerts(evaluate_equipment(row["Equipment ID"],
                                                        row["Temperature"],
                                                        row["Vibration"])):
            print(alert)
            print()

Equipment ID  Temperature  Vibration Temp Status Vib Status Condition
     TRK-001           72        3.1      NORMAL     NORMAL    NORMAL
     TRK-012           91        9.2     WARNING   CRITICAL  CRITICAL
     LDR-004          105        4.0    CRITICAL     NORMAL  CRITICAL
     CNV-002           60        6.5      NORMAL    WARNING   WARNING

{'total': 4, 'normal': 1, 'warning': 1, 'critical': 2, 'invalid': 0, 'percent_normal': 25.0}

MINING MONITORING ALERT
[WARNING]
Equipment ID: TRK-012
Parameter: Temperature
Value: 91.0 °C
Status: Approaching Critical Threshold
Recommended Action:
 Monitor closely and schedule an inspection.

MINING MONITORING ALERT
[CRITICAL]
Equipment ID: TRK-012
Parameter: Vibration
Value: 9.2 mm/s
Status: Above Critical Threshold
Recommended Action:
 Inspect equipment and initiate appropriate
 maintenance/safety procedures.

MINING MONITORING ALERT
[CRITICAL]
Equipment ID: LDR-004
Parameter: Temperature
Value: 105.0 °C
Status: Above Critical Threshold
Rec